# 00 — Admission-level data audit

**Purpose.** Check the one-row-per-admission input before cohort splitting or model training. This notebook reports problems; it does not silently remove rows or alter predictor values.

**Important assumptions**
- The input contains one row per admission.
- The configured target is the outcome for supervised modelling.
- Missing or `unknown` outcomes remain visible in the audit and are excluded later during model fitting.
- Re-run this notebook after changing the input path or column definitions.


In [42]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
from pathlib import Path
import yaml
import numpy as np
from icare_risk.data.audit import (
    load_admission_file, dataset_overview, column_summary,
    duplicate_summary, outcome_summary, temporal_summary, save_audit_report,
)
from icare_risk.data.validation import validate_admission_table, validate_binary_target

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)

input_cfg = config['input']
columns = config['columns']
target_mapping = config.get('target_mapping', {})

print('Configuration:')
print(f"Input: {input_cfg['path']}")
print(f"Admission ID: {columns['admission_id']}")
print(f"Patient ID: {columns['patient_id']}")
print(f"Admission date: {columns['admission_date']}")
print(f"Target: {columns['target']}")
print(f"Positive labels: {target_mapping.get('positive_values', [])}")
print(f"Negative labels: {target_mapping.get('negative_values', [])}")

Configuration:
Input: ../notebooks/data/synthetic/10000p/03_derived_analysis_dataset.parquet
Admission ID: SPELL_IDENTIFIER
Patient ID: SUBJECT
Admission date: ADMISSION_DATE
Target: esbl_label_test
Positive labels: ['esbl', 'positive', 1, '1']
Negative labels: ['non-esbl', 'non esbl', 'negative', 0, '0']


In [45]:
df = load_admission_file(
    path=input_cfg['path'],
    **input_cfg.get('read_csv_kwargs', {})
)
print(f"Loaded {len(df):,} rows and {df.shape[1]:,} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(df.head())

Loaded 10,000 rows and 186 columns
Memory usage: 21.25 MB


,SUBJECT,SPELL_IDENTIFIER,ENCNTR_ID,EPISODE_IDENTIFIER,ADMISSION_DATE,ADMISSION_TIME,AGE_AT_ADMISSION,DISCHARGE_DATE,MAIN_SPECIALTY_CODE,MAIN_SPECIALTY_CODE_DESC,...,time_to_culture_hours,esbl_status,BIRTH_DATE,DEATH_DATE,ETHNICITY,ETHNICITY_DES,GENDER,GENDER_DESC,UPDATE_DT_TM,esbl_label_test
0,10001,7929089,2005940,4725136,2020-06-10,22:22:00,35,2020-07-03,320,cardiology,...,NaN,unknown,1981-01-23,None,A,White - British,U,Unknown,2023-03-28,0
1,10002,8200425,6570033,6896867,2020-03-27,17:31:00,63,2020-03-28,301,gastroenterology,...,NaN,unknown,1956-11-10,None,L,Any other Asian background,U,Unknown,2023-03-26,0
2,10003,9931252,5492977,1426665,2023-01-06,00:18:00,89,2023-01-25,301,gastroenterology,...,NaN,unknown,1965-05-14,None,N,Black or Black British - African,I,Indeterminate,2023-04-04,0
3,10004,5095152,3730006,3105590,2022-09-14,10:32:00,25,2022-10-12,301,gastroenterology,...,NaN,unknown,1980-03-11,None,L,Any other Asian background,U,Unknown,2023-05-10,0
4,10005,7807609,7413177,1104787,2023-09-08,10:55:00,80,2023-09-23,140,oral surgery,...,NaN,unknown,1977-12-18,None,Z,Not stated,M,Male,2023-07-14,0


In [46]:
print(
    "\n" + "!" * 80 +
    "\nWARNING: esbl_label_test IS BEING MANUALLY GENERATED FOR DEMOS ONLY!"
    "\nThese values are randomly generated and must NOT be used for real analysis."
    "\n" + "!" * 80 + "\n"
)

df["esbl_label_test"] = np.random.choice(
    [0, 1],
    size=len(df),
    p=[0.8, 0.2]
)

df.to_parquet(input_cfg['path'])


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
These values are randomly generated and must NOT be used for real analysis.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



In [47]:
required = config['validation']['required_columns']
structure_validation = validate_admission_table(
    df,
    required_columns=required,
    admission_id=columns['admission_id'],
    date_columns=[
        columns['admission_date'],
        columns['discharge_date']
    ],
)
target_validation = validate_binary_target(df, columns['target'])

print('Structural validation')
display(structure_validation)
print('\nTarget validation')
display(target_validation)
print('\nObserved target values')
display(df[columns['target']] \
        .value_counts(dropna=False) \
        .rename_axis('value') \
        .reset_index(name='n'))

Structural validation


{'n_rows': 10000,
 'n_columns': 186,
 'is_empty': False,
 'admission_id_present': True,
 'duplicate_admission_rows': 14,
 'date_parse_failures': {'ADMISSION_DATE': 0, 'DISCHARGE_DATE': 0},
 'warnings': ['Duplicate values found in SPELL_IDENTIFIER']}


Target validation


{'target_column': 'esbl_label_test',
 'n_missing': 0,
 'unique_values': [1, 0],
 'n_unique_non_missing': 2,
 'is_binary': True}


Observed target values


,value,n
0,0,8042
1,1,1958


In [48]:
report = {
    'dataset_overview': dataset_overview(df),
    'column_summary': column_summary(df),
    'duplicate_summary': duplicate_summary(df, [columns['admission_id'], columns['patient_id']]),
    'outcome_summary': outcome_summary(df, columns['target']),
    'temporal_summary': temporal_summary(
        df,
        date_column=columns['admission_date'],
        target_column=columns['target'],
        freq=config['analysis']['temporal_frequency'],
        #target_mapping=target_mapping,
    ),
}
for name, table in report.items():
    print(f'\n--- {name} ---')
    display(table.head(20))

save_audit_report(report, config['outputs']['audit_directory'])
print(f"Saved audit tables to {config['outputs']['audit_directory']}")


--- dataset_overview ---


,n_rows,n_columns,n_subjects,n_admissions,memory_mb
0,10000,186,10000,9993,21.25



--- column_summary ---


,column,dtype,n_rows,n_missing,missing_pct,n_unique,example_values
172,days_since_last_discharge,float64,10000,10000,100.00,0,
173,qualifying_last_abx,object,10000,10000,100.00,0,
124,charlson_hx_leukemia_dsfirst,float64,10000,9997,99.97,3,"547.1208333333333, 823.4736111111112, 324.6111..."
123,charlson_hx_leukemia_dslast,float64,10000,9997,99.97,3,"547.1208333333333, 823.4736111111112, 324.6111..."
133,charlson_hx_cancer_met_dsfirst,float64,10000,9996,99.96,4,"942.3263888888889, 1172.9541666666667, 284.070..."
132,charlson_hx_cancer_met_dslast,float64,10000,9996,99.96,4,"942.3263888888889, 1172.9541666666667, 284.070..."
121,charlson_hx_cancer_solid_dsfirst,float64,10000,9996,99.96,4,"942.3263888888889, 1172.9541666666667, 284.070..."
120,charlson_hx_cancer_solid_dslast,float64,10000,9996,99.96,4,"942.3263888888889, 1172.9541666666667, 284.070..."
154,gavaghan_hx_urinary_catheter_dsfirst,float64,10000,9994,99.94,6,"899.4729166666667, 649.0027777777777, 410.1409..."
153,gavaghan_hx_urinary_catheter_dslast,float64,10000,9994,99.94,6,"899.4729166666667, 649.0027777777777, 410.1409..."



--- duplicate_summary ---


...


--- outcome_summary ---


,value,n,percentage
0,0,8042,80.42
1,1,1958,19.58



--- temporal_summary ---


,period,n_admissions,n_events,n_missing_outcome,event_rate
0,2020-01-01,2497,479.0,0,0.19183
1,2021-01-01,2521,509.0,0,0.201904
2,2022-01-01,2505,477.0,0,0.190419
3,2023-01-01,2472,493.0,0,0.199434
4,2024-01-01,5,0.0,0,0.0


Saved audit tables to results/audit
